# Llama-2-7B FSDP SFT on StarCoder Go+Rust via torchrun + accelerate

**What `torchrun` and `hf_accelerate` are.** Thin launchers. Unlike
the opinionated frameworks (slime, verl, ms-swift, megatron), these
don't wrap a specific training library — they just spawn **your
Python script** under `torchrun` / `accelerate launch` across a
Modal cluster, with NCCL initialized and RDMA configured. Bring
your own training loop.

**What this tutorial runs.** Supervised fine-tuning of Llama-2-7B on
Go + Rust shards of
[`bigcode/starcoderdata`](https://huggingface.co/datasets/bigcode/starcoderdata),
using PyTorch FSDP (`full_shard auto_wrap`), activation checkpointing,
and BF16 mixed precision. The same script runs under both launchers
— the comparison is the point: launcher choice is independent of
training code, and `accelerate` mainly adds the
`--mixed_precision bf16` wrapper on top of `torchrun`'s spawn
semantics.

**What makes it non-trivial.** The training script is shipped as a
Python *source string* and written into a volume by `upload_script`
before `train` consumes it. This lets you iterate on the script from
a notebook cell (edit the string, re-upload, re-run) without
rebuilding the container image. Same pattern both launchers use.

For the shared primitives (`DatasetConfig`, `Model`, `build_app()`)
see [`quickstart`](../../intro/quickstart/quickstart.ipynb).

**What you'll need.**
- Access to Modal's multi-node training preview (2 × 8×H100).
- HF access to `meta-llama/Llama-2-7b-hf` (gated).
- `huggingface-secret` + `wandb-secret` Modal secrets.

**What to watch.** W&B project `bigcode-starcoderdata-training`.
Loss on code SFT drops steadily; watch for FSDP OOMs in the first
few steps if you crank `batch_per_device`.

In [ ]:
! pip install -q git+https://github.com/modal-projects/training-gym.git@joy/initial-setup

In [ ]:
import modal

from modal_training_gym.common.dataset import DatasetConfig
from modal_training_gym.common.models import Llama2_7B
from modal_training_gym.common.wandb import WandbConfig
from modal_training_gym.frameworks.hf_accelerate import (
    AccelerateConfig,
    AccelerateFrameworkConfig,
)
from modal_training_gym.frameworks.torchrun import (
    DATASET_MOUNT_PATH,
    TorchrunConfig,
    TorchrunFrameworkConfig,
)

## Define the dataset

StarCoder is split into many language shards. We want Go + Rust
only, and have to filter out a few incompatible shard families
(jupyter scripts, github issues, git commits) that the training
script would choke on. `prepare()` lists the repo files, matches
`{go,rust}/**/*.parquet`, excludes the known-bad ones, and
`save_to_disk`-es each shard into the data volume as an `.arrow`
file. The training script globs `go/**/*.arrow` + `rust/**/*.arrow`
under `--data_dir`.

In [ ]:
class StarcoderGoRustDataset(DatasetConfig):
    """StarCoder: Go + Rust shards from `bigcode/starcoderdata`."""

    def __init__(self, data_path):
        self._data_path = str(data_path)
        self.prompt_data = self._data_path

    def prepare(self):
        import os
        from datasets import load_dataset
        from huggingface_hub import HfApi

        api = HfApi()
        all_paths = api.list_repo_files(
            repo_id="bigcode/starcoderdata", repo_type="dataset"
        )
        excluded = (
            "jupyter-scripts-dedup-filtered",
            "jupyter-structured-clean-dedup",
            "github-issues-filtered-structured",
            "git-commits-cleaned",
        )
        file_paths = [
            p
            for p in all_paths
            if ".parquet" in p
            and any(p.startswith(f"{lang}/") for lang in ("go", "rust"))
            and not any(x in p for x in excluded)
        ]
        print(f"Downloading {len(file_paths)} shards…")
        for fp in file_paths:
            save_path = f"{self._data_path}/{fp}"
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            ds = load_dataset("bigcode/starcoderdata", data_files=fp)
            ds.save_to_disk(save_path)
        print(f"Saved {len(file_paths)} shards to {self._data_path}")

## Define the training script

The script is a plain Python module defined as a multi-line source
string. Both launchers ship it the same way — the framework's
`upload_script` writes it into a scripts volume that `train`
mounts. The script itself:

- Downloads Llama-2-7B + tokenizer with `AutoModelForCausalLM`.
- Builds a packed token stream from the arrow files using an
  `IterableDataset`; each block is `block_size` tokens glued
  together with EOS separators.
- Runs TRL's `SFTTrainer` with `fsdp="full_shard auto_wrap"` +
  `activation_checkpointing=True`.
- Disables `model.config.use_cache` — KV cache breaks activation
  checkpointing.

Editing the script: change `TRAIN_SCRIPT` below, re-run
`upload_script`, then re-run `train`. No image rebuild needed.

In [ ]:
import textwrap

TRAIN_SCRIPT = textwrap.dedent(r'''
    import argparse
    import os
    from pathlib import Path

    import datasets
    import torch
    import torch.distributed as dist
    from datasets import load_dataset
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        DataCollatorForLanguageModeling,
    )
    from trl import SFTConfig, SFTTrainer

    MODEL_NAME = "meta-llama/Llama-2-7b-hf"


    def download_llama2(cache_dir):
        token = os.environ["HF_TOKEN"]
        tok = AutoTokenizer.from_pretrained(MODEL_NAME, token=token, cache_dir=cache_dir)
        tok.pad_token = tok.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, torch_dtype="auto", token=token, cache_dir=cache_dir,
        )
        model.config.use_cache = False  # KV cache breaks activation checkpointing
        return model, tok


    def build_packed_ds(data_dir, tokenizer, buffer_size, block):
        eos_id = tokenizer.eos_token_id

        def gen():
            files = []
            for lang_dir in ["go", "rust"]:
                files.extend(str(f) for f in (data_dir / lang_dir).glob("**/*.arrow"))
            print(f"Found {len(files)} arrow files")
            ds = load_dataset("arrow", data_files=files, split="train", streaming=True)
            ds = ds.shuffle(buffer_size=buffer_size, seed=44)

            buf = []
            for rec in ds:
                buf.extend(
                    tokenizer(rec["content"], add_special_tokens=False)["input_ids"]
                    + [eos_id]
                )
                while len(buf) >= block:
                    yield {"input_ids": buf[:block], "attention_mask": [1] * block}
                    del buf[:block]

        return datasets.IterableDataset.from_generator(gen)


    def parse_args():
        p = argparse.ArgumentParser()
        p.add_argument("--data_dir", required=True)
        p.add_argument("--output_dir", required=True)
        p.add_argument("--model_cache_dir", default=None)
        p.add_argument("--epochs", type=int, default=2)
        p.add_argument("--batch_per_device", type=int, default=16)
        p.add_argument("--grad_accum", type=int, default=2)
        p.add_argument("--block_size", type=int, default=4096)
        return p.parse_args()


    def main():
        args = parse_args()

        if not dist.is_initialized():
            dist.init_process_group(
                backend="nccl",
                device_id=torch.device(f"cuda:{os.environ['LOCAL_RANK']}"),
            )

        model, tokenizer = download_llama2(args.model_cache_dir)
        train_ds = build_packed_ds(
            Path(args.data_dir), tokenizer, buffer_size=20_000, block=args.block_size,
        )
        collator = DataCollatorForLanguageModeling(tokenizer, mlm=False, pad_to_multiple_of=8)

        cfg = SFTConfig(
            output_dir=args.output_dir,
            seed=1234,
            per_device_train_batch_size=args.batch_per_device,
            gradient_accumulation_steps=args.grad_accum,
            learning_rate=8e-5,
            lr_scheduler_type="cosine",
            warmup_ratio=0.03,
            bf16=True,
            max_seq_length=args.block_size,
            save_steps=125,
            logging_steps=1,
            report_to="wandb" if os.environ.get("WANDB_PROJECT") else "none",
            run_name=os.environ.get("WANDB_RUN_NAME"),
            fsdp="full_shard auto_wrap",
            fsdp_config={
                "activation_checkpointing": True,
                "forward_prefetch": False,
                "fsdp_transformer_layer_cls_to_wrap": ["LlamaDecoderLayer"],
            },
            num_train_epochs=args.epochs,
            max_steps=1,
        )

        SFTTrainer(
            model=model,
            train_dataset=train_ds,
            args=cfg,
            data_collator=collator,
        ).train()

        if dist.is_initialized():
            dist.destroy_process_group()


    if __name__ == "__main__":
        main()
''').lstrip("\n")

## Define the run config

Both launchers share the same model, dataset, and W&B config. The
full reference recipe (not this smoke run) uses:

| Parameter | Value |
|---|---|
| Global batch size | 2048 |
| Per-device batch size | 16 |
| Gradient accumulation | 2 |
| Learning rate | 8e-5, cosine decay |
| Context length | 4096 |
| Epochs | 2 |

Below we override to a 1-step smoke run (`--block_size 512`,
`--batch_per_device 1`, `--epochs 1`) so the tutorial actually
completes quickly. Bump the `_SHARED_SCRIPT_ARGS` to the reference
values for a real run.

In [ ]:
_MODEL = Llama2_7B()
_DATASET = StarcoderGoRustDataset(DATASET_MOUNT_PATH)
_WANDB = WandbConfig(project="bigcode-starcoderdata-training")

_SHARED_SCRIPT_ARGS = [
    "--epochs", "1",
    "--batch_per_device", "1",
    "--grad_accum", "1",
    "--block_size", "512",
    "--model_cache_dir", "/model/model_cache",
]

## Launcher A: `torchrun`

Stock PyTorch distributed launcher — directly invokes
`torch.distributed.run.run(...)` on each node. Nothing added, nothing
hidden.

In [ ]:
torchrun_framework_config = TorchrunFrameworkConfig(
    gpu="H100",
    n_nodes=2,
    gpus_per_node=8,
    train_script_source=TRAIN_SCRIPT,
    script_args=_SHARED_SCRIPT_ARGS,
)

torchrun_run = TorchrunConfig(
    dataset=_DATASET,
    model=_MODEL,
    wandb=_WANDB,
    framework_config=torchrun_framework_config,
)

torchrun_app = torchrun_run.build_app()

## Launcher B: `hf_accelerate`

Same script, launched via `accelerate launch`. Adds
`--mixed_precision bf16` on top of torchrun's spawn semantics and
reads HF's `accelerate` config where relevant; otherwise identical.

In [ ]:
accelerate_framework_config = AccelerateFrameworkConfig(
    gpu="H100",
    n_nodes=2,
    gpus_per_node=8,
    train_script_source=TRAIN_SCRIPT,
    script_args=_SHARED_SCRIPT_ARGS,
    mixed_precision="bf16",
)

accelerate_run = AccelerateConfig(
    dataset=_DATASET,
    model=_MODEL,
    wandb=_WANDB,
    framework_config=accelerate_framework_config,
)

accelerate_app = accelerate_run.build_app()

## Run it

Both apps have the same three functions:

- `download_dataset` — runs `StarcoderGoRustDataset.prepare()`.
- `upload_script` — writes `TRAIN_SCRIPT` into the scripts volume.
- `train` — clustered multi-node; launches the uploaded script.

Each app has its own volumes (`<app_name>-data`, `-model`,
`-scripts`), so running torchrun and accelerate side-by-side
doesn't cross-contaminate state.

Pick one launcher and run the three stages in order.

### Interactive — torchrun

In [ ]:
with torchrun_app.run():
    torchrun_app.download_dataset.remote()

In [ ]:
with torchrun_app.run():
    torchrun_app.upload_script.remote()

In [ ]:
with torchrun_app.run():
    torchrun_app.train.remote()

### Interactive — accelerate

In [ ]:
with accelerate_app.run():
    accelerate_app.download_dataset.remote()

In [ ]:
with accelerate_app.run():
    accelerate_app.upload_script.remote()

In [ ]:
with accelerate_app.run():
    accelerate_app.train.remote()

## Scaling

Measured sample consumption scales sub-linearly with node count at
these parameters. Increasing the global batch size and tuning FSDP
shard/prefetch settings can recover efficiency at higher scale.

| Nodes | GPUs/node | Samples/min | Tokens/min |
|---|---|---|---|
| 2 | 8 | 2,898 | 6,151,645 |
| 4 | 8 | 4,981 | 10,625,570 |
| 8 | 8 | 7,675 | 16,357,785 |